# 02: Feature Matrix Construction

**Purpose:** Build the cleaned, model-ready feature matrix from the 7 raw CSV files in `data/raw/`. 
 
**Output:** `data/processed/features.csv` and `data/processed/labels.csv`

This notebook implements the **Feature Plan** defined in the final cell of `notebooks/01_EDA.ipynb`.
It follows the same label derivation logic as EDA Section 3 and applies all exclusions from Section C of the Feature Plan.

---

### Contents
1. [Imports and Configuration](#1)
2. [Load Raw Data](#2)
3. [Derive Target Variable (`copd_label`)](#3)
4. [Section A: Original Variables](#4)
5. [Section B1: Spirometry Engineered Features](#5)
6. [Section B2: Habits and Labs: Flags and Interactions](#6)
7. [Section B3: ICD-10 Comorbidity Flags](#7)
8. [Section B4: NLP Keyword Extraction](#8)
9. [Merge Feature Groups](#9)
10. [Missing Value Analysis](#10)
11. [Imputation and Encoding](#11)
12. [Final Feature Matrix Validation](#12)
13. [Save Outputs](#13)

---
## 1. Imports and Configuration <a id='1'></a>

In [27]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

# ── Paths ───────────────────────────────────────────────────────────────────
ROOT     = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
RAW      = ROOT / 'data' / 'raw'
PROC     = ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

# ── Label threshold (GOLD 2024) ──────────────────────────────────────────────
THRESHOLD = 0.70

# ── Reproducibility ─────────────────────────────────────────────────────────
RANDOM_STATE = 42

print(f'ROOT : {ROOT}')
print(f'RAW  : {RAW}')
print(f'PROC : {PROC}')
print(f'Label threshold: FEV1/FVC < {THRESHOLD}')

ROOT : c:\Users\jorge\OneDrive - IE University\Documentos\MBDS2025\Biopharma Specialization\Project\Dev
RAW  : c:\Users\jorge\OneDrive - IE University\Documentos\MBDS2025\Biopharma Specialization\Project\Dev\data\raw
PROC : c:\Users\jorge\OneDrive - IE University\Documentos\MBDS2025\Biopharma Specialization\Project\Dev\data\processed
Label threshold: FEV1/FVC < 0.7


---
## 2. Load Raw Data <a id='2'></a>

In [28]:
patients         = pd.read_csv(RAW / 'patients.csv')
habits           = pd.read_csv(RAW / 'habits.csv')
lab_results      = pd.read_csv(RAW / 'lab_results.csv')
spirometry       = pd.read_csv(RAW / 'spirometry.csv', parse_dates=['fecha'])
clinical_events  = pd.read_csv(RAW / 'clinical_events.csv')
clinical_notes   = pd.read_csv(RAW / 'clinical_notes.csv')

print('Raw data loaded:')
for name, df in [
    ('patients',        patients),
    ('habits',          habits),
    ('lab_results',     lab_results),
    ('spirometry',      spirometry),
    ('clinical_events', clinical_events),
    ('clinical_notes',  clinical_notes),
]:
    print(f'  {name:<20} {df.shape[0]:>6} rows  {df.shape[1]:>2} cols')

Raw data loaded:
  patients              10000 rows   6 cols
  habits                10000 rows   5 cols
  lab_results           10000 rows   7 cols
  spirometry            20068 rows   5 cols
  clinical_events       12073 rows   3 cols
  clinical_notes        10000 rows   2 cols


---
## 3. Derive Target Variable (`copd_label`) <a id='3'></a>

**Definition:** `copd_label = 1` if the patient's minimum FEV1/FVC ratio across all valid spirometry tests is < 0.70.

This is the GOLD 2024 criterion for airflow obstruction. Using `MIN` rather than requiring ALL tests to be below threshold is clinically correct: COPD is irreversible, so a single valid obstructive reading is sufficient for classification.

**Data quality step:** Rows with FEV1/FVC > 1.0 are physically impossible (FEV1 cannot exceed FVC) and are removed before label derivation.

In [29]:
# ── Clean spirometry: remove physically impossible rows ──────────────────────
n_raw = len(spirometry)
spirometry_clean = spirometry[
    (spirometry['fev1_fvc_ratio'] <= 1.0) &
    (spirometry['fev1_fvc_ratio'] >= 0.10)
].copy()
n_removed = n_raw - len(spirometry_clean)
print(f'Spirometry rows: {n_raw} raw → {len(spirometry_clean)} clean ({n_removed} removed, ratio outside [0.10, 1.00])')

# ── Aggregate to one row per patient ─────────────────────────────────────────
spiro_agg = (
    spirometry_clean
    .sort_values(['id_paciente', 'fecha'])
    .groupby('id_paciente')
    .agg(
        spiro_min_ratio    = ('fev1_fvc_ratio', 'min'),
        spiro_latest_ratio = ('fev1_fvc_ratio', 'last'),
        spiro_first_ratio  = ('fev1_fvc_ratio', 'first'),
        spiro_mean_fev1    = ('fev1', 'mean'),
        spiro_mean_fvc     = ('fvc', 'mean'),
        spiro_n_tests      = ('fev1_fvc_ratio', 'count'),
        spiro_first_date   = ('fecha', 'first'),
        spiro_last_date    = ('fecha', 'last'),
    )
    .reset_index()
)

# ── Longitudinal features ─────────────────────────────────────────────────────
spiro_agg['spiro_years_span'] = (
    (spiro_agg['spiro_last_date'] - spiro_agg['spiro_first_date']).dt.days / 365.25
)
spiro_agg['spiro_slope'] = np.where(
    (spiro_agg['spiro_n_tests'] >= 2) & (spiro_agg['spiro_years_span'] > 0),
    (spiro_agg['spiro_latest_ratio'] - spiro_agg['spiro_first_ratio']) / spiro_agg['spiro_years_span'],
    np.nan
)

# ── Derive label ──────────────────────────────────────────────────────────────
spiro_agg['copd_label'] = (spiro_agg['spiro_min_ratio'] < THRESHOLD).astype(int)

n_patients_spiro = len(spiro_agg)
n_copd           = spiro_agg['copd_label'].sum()
prevalence       = n_copd / n_patients_spiro * 100
imbalance        = (n_patients_spiro - n_copd) / n_copd

print(f'\nPatients with spirometry: {n_patients_spiro:,}')
print(f'COPD positive (label=1) : {n_copd:,} ({prevalence:.1f}%)')
print(f'COPD negative (label=0) : {n_patients_spiro - n_copd:,} ({100-prevalence:.1f}%)')
print(f'Class imbalance ratio   : {imbalance:.1f}:1  (negative:positive)')

Spirometry rows: 20068 raw → 19713 clean (355 removed, ratio outside [0.10, 1.00])

Patients with spirometry: 9,948
COPD positive (label=1) : 7,600 (76.4%)
COPD negative (label=0) : 2,348 (23.6%)
Class imbalance ratio   : 0.3:1  (negative:positive)


---
## 4. Section A: Original Variables <a id='4'></a>

Raw columns from `patients.csv`, `habits.csv`, and `lab_results.csv`, used without transformation. These features require only imputation and scaling, both applied later in the pipeline.

**Exclusions:**
- `nivel_socioeconomico`, `zona_residencia`: excluded due to bias risk (proxies for healthcare access, not disease biology)

**Note: `grupo_sanguineo`:** Loaded here but one-hot encoded in Section 11. It is kept in `features.csv` for the sensitivity analysis variant but excluded from the main model feature list (defined as `FEATURES_MAIN` in the training script).

In [30]:
# ── Patients: demographics ────────────────────────────────────────────────────
PATIENT_COLS = ['id_paciente', 'edad', 'sexo', 'imc']
patients_sel = patients[PATIENT_COLS].copy()

# ── Habits: smoking and activity ──────────────────────────────────────────────
HABIT_COLS = ['id_paciente', 'fumador_actual', 'exfumador', 'paquetes_ano', 'actividad_fisica']
habits_sel = habits[HABIT_COLS].copy()

# ── Lab results ───────────────────────────────────────────────────────────────
LAB_COLS = ['id_paciente', 'pcr_mg_l', 'vitamina_d_ng_ml', 'colesterol_total_mg_dl',
            'ferritina_ng_ml', 'tsh_ui_ml', 'grupo_sanguineo']  # kept for sensitivity analysis
labs_sel = lab_results[LAB_COLS].copy()

print('Section A variable counts:')
print(f'  Demographics : {len(PATIENT_COLS)-1} variables')
print(f'  Habits       : {len(HABIT_COLS)-1} variables')
print(f'  Labs         : {len(LAB_COLS)-1} variables')
print(f'  Section A total: {len(PATIENT_COLS) + len(HABIT_COLS) + len(LAB_COLS) - 3} variables')

Section A variable counts:
  Demographics : 3 variables
  Habits       : 4 variables
  Labs         : 6 variables
  Section A total: 13 variables


---
## 5. Section B1: Spirometry Engineered Features <a id='5'></a>

Derived from the longitudinal aggregation performed in Section 3.

**Critical exclusion:** `spiro_min_ratio` is the exact basis of `copd_label`, so including it as a feature is direct data leakage. `spiro_first_ratio` and the raw date columns are also excluded (transformed into slope/span).

In [31]:
SPIRO_FEATURE_COLS = [
    'id_paciente',
    'spiro_latest_ratio',   # current lung function status
    'spiro_slope',          # annual FEV1/FVC change rate (negative = progressive deterioration)
    'spiro_n_tests',        # follow-up intensity
    'spiro_years_span',     # observation window
    'spiro_mean_fev1',      # absolute lung volume
    'spiro_mean_fvc',       # differentiates obstructive vs restrictive pattern
]

# spiro_min_ratio EXCLUDED: label leakage
# spiro_first_ratio EXCLUDED: absorbed into spiro_slope
# spiro_first_date, spiro_last_date EXCLUDED: absorbed into spiro_years_span

spiro_features = spiro_agg[SPIRO_FEATURE_COLS].copy()

print('Section B1: Spirometry engineered features:')
print(spiro_features[SPIRO_FEATURE_COLS[1:]].describe().round(3).to_string())

Section B1: Spirometry engineered features:
       spiro_latest_ratio  spiro_slope  spiro_n_tests  spiro_years_span  spiro_mean_fev1  spiro_mean_fvc
count            9948.000     6573.000       9948.000          9948.000         9948.000        9948.000
mean                0.659       -0.007          1.982             1.095            1.688           2.515
std                 0.151        2.387          0.812             1.111            0.581           0.649
min                 0.210      -76.702          1.000             0.000            0.500           1.000
25%                 0.560       -0.087          1.000             0.000            1.290           2.070
50%                 0.670        0.003          2.000             0.827            1.673           2.490
75%                 0.760        0.095          3.000             1.972            2.060           2.933
max                 1.000      124.185          3.000             3.959            4.320           5.170


---
## 6. Section B2: Habits and Labs: Flags and Interactions <a id='6'></a>

Binary flags and interaction terms derived from `habits.csv` and `lab_results.csv`.

Three clinical thresholds are applied: `ever_smoked` captures any smoking history (the dominant modifiable risk factor for COPD); `crp_elevated` flags systemic inflammation (CRP > 10 mg/L); `vitamin_d_deficient` flags a deficiency linked to accelerated lung function decline. The `age_x_packyears` interaction term captures the compound effect of cumulative smoke exposure over a longer life. Two patients with identical pack-year counts carry different risk profiles at age 40 vs. 70.

In [32]:
# ── Merge habits and labs for engineering ─────────────────────────────────────
hl = habits_sel.merge(labs_sel, on='id_paciente', how='left')
hl = hl.merge(patients_sel[['id_paciente', 'edad']], on='id_paciente', how='left')

# ── Binary flags ──────────────────────────────────────────────────────────────
hl['ever_smoked']        = ((hl['fumador_actual'] == 1) | (hl['exfumador'] == 1)).astype(int)
hl['crp_elevated']       = (hl['pcr_mg_l'] > 10).astype(int)          # clinically significant inflammation
hl['vitamin_d_deficient']= (hl['vitamina_d_ng_ml'] < 20).astype(int)  # deficiency threshold

# ── Interaction term ──────────────────────────────────────────────────────────
hl['age_x_packyears']    = hl['edad'] * hl['paquetes_ano']             # compounded risk

ENGINEERED_HL_COLS = ['id_paciente', 'ever_smoked', 'crp_elevated', 'vitamin_d_deficient', 'age_x_packyears']
hl_features = hl[ENGINEERED_HL_COLS].copy()

print('Section B2: Habits/labs engineered features:')
print(f'  ever_smoked         : {hl_features["ever_smoked"].sum():,} patients (flag=1)')
print(f'  crp_elevated        : {hl_features["crp_elevated"].sum():,} patients (CRP > 10 mg/L)')
print(f'  vitamin_d_deficient : {hl_features["vitamin_d_deficient"].sum():,} patients (Vit D < 20 ng/mL)')
print(f'  age_x_packyears     : mean = {hl_features["age_x_packyears"].mean():.1f}, max = {hl_features["age_x_packyears"].max():.1f}')

Section B2: Habits/labs engineered features:
  ever_smoked         : 4,944 patients (flag=1)
  crp_elevated        : 85 patients (CRP > 10 mg/L)
  vitamin_d_deficient : 2,715 patients (Vit D < 20 ng/mL)
  age_x_packyears     : mean = 527.0, max = 8109.0


---
## 7. Section B3: ICD-10 Comorbidity Flags <a id='7'></a>

One binary flag per relevant ICD-10 code from `clinical_events.csv`.

**Critical exclusion:** ICD J44 (COPD diagnosis) is excluded. It is the administrative encoding of the label and constitutes direct data leakage.

In [33]:
# ── ICD-10 code mapping ───────────────────────────────────────────────────────
# J44 (COPD) is deliberately EXCLUDED: label leakage
ICD_MAP = {
    'J45': 'has_asthma',       # Asthma-COPD overlap syndrome
    'I10': 'has_hypertension',  # Cardiovascular comorbidity
    'E11': 'has_diabetes',      # Metabolic comorbidity
    'I25': 'has_ihd',           # Ischaemic heart disease
    'F32': 'has_depression',    # 2-3x higher rate in COPD patients
}

# ── Count distinct diagnoses per patient (comorbidity burden) ─────────────────
# Exclude J44 from the burden count to avoid leakage
non_leakage_events = clinical_events[clinical_events['cie10'] != 'J44']
n_diag = (
    non_leakage_events.groupby('id_paciente')['cie10']
    .nunique()
    .reset_index()
    .rename(columns={'cie10': 'n_unique_diag'})
)

# ── Build pivot table: one binary column per ICD code ────────────────────────
icd_flags = pd.DataFrame({'id_paciente': patients['id_paciente'].unique()})
for code, col_name in ICD_MAP.items():
    has_code = clinical_events[clinical_events['cie10'] == code]['id_paciente'].unique()
    icd_flags[col_name] = icd_flags['id_paciente'].isin(has_code).astype(int)

icd_flags = icd_flags.merge(n_diag, on='id_paciente', how='left')
icd_flags['n_unique_diag'] = icd_flags['n_unique_diag'].fillna(0).astype(int)

print('Section B3: ICD-10 comorbidity flags:')
for code, col in ICD_MAP.items():
    n = icd_flags[col].sum()
    print(f'  {code} {col:<20} : {n:,} patients ({n/len(icd_flags)*100:.1f}%)')
print(f'  n_unique_diag        : mean = {icd_flags["n_unique_diag"].mean():.2f}')

Section B3: ICD-10 comorbidity flags:
  J45 has_asthma           : 2,134 patients (21.3%)
  I10 has_hypertension     : 2,600 patients (26.0%)
  E11 has_diabetes         : 1,640 patients (16.4%)
  I25 has_ihd              : 1,639 patients (16.4%)
  F32 has_depression       : 1,192 patients (11.9%)
  n_unique_diag        : mean = 0.92


---
## 8. Section B4: NLP Keyword Extraction <a id='8'></a>

Binary flags and a composite score from Spanish-language clinical notes in `clinical_notes.csv`.

**Critical exclusion:** `kw_copd_mention` (terms 'EPOC', 'enfermedad pulmonar obstructiva') is excluded. Notes mentioning COPD by name are written *after* diagnosis, constituting temporal leakage.

**Also excluded:** `kw_smoking`: redundant with `fumador_actual`, `exfumador`, and `paquetes_ano`.

In [34]:
# ── Keyword patterns (Spanish) ────────────────────────────────────────────────
KEYWORD_PATTERNS = {
    'kw_dyspnea'     : r'disnea|ahogo|falta de aire',
    'kw_cough'       : r'tos cr.nica|tos persistente|tos',
    'kw_sputum'      : r'esputo|expectoraci.n',
    'kw_exacerbation': r'exacerbaci.n|agudizaci.n',
    'kw_breathless'  : r'dificultad respiratoria|disnea de esfuerzo',
    'kw_no_symptoms' : r'sin s.ntomas|asintom.tico',
    'kw_wheezing'    : r'sibilancias|pitidos',
}

notes = clinical_notes.copy()
notes['nota_lower'] = notes['nota_clinica'].str.lower().fillna('')

for col, pattern in KEYWORD_PATTERNS.items():
    notes[col] = notes['nota_lower'].str.contains(pattern, regex=True, na=False).astype(int)

# ── Composite symptom burden score ────────────────────────────────────────────
positive_kw = [k for k in KEYWORD_PATTERNS if k != 'kw_no_symptoms']
notes['keyword_score'] = notes[positive_kw].sum(axis=1)

NLP_COLS = ['id_paciente'] + list(KEYWORD_PATTERNS.keys()) + ['keyword_score']
nlp_features = notes[NLP_COLS].copy()

print('Section B4: NLP keyword extraction:')
for col in list(KEYWORD_PATTERNS.keys()):
    n = nlp_features[col].sum()
    print(f'  {col:<22}: {n:,} patients ({n/len(nlp_features)*100:.1f}%)')
print(f'  keyword_score        : mean = {nlp_features["keyword_score"].mean():.2f}, max = {nlp_features["keyword_score"].max()}')

Section B4: NLP keyword extraction:
  kw_dyspnea            : 2,974 patients (29.7%)
  kw_cough              : 0 patients (0.0%)
  kw_sputum             : 0 patients (0.0%)
  kw_exacerbation       : 2,528 patients (25.3%)
  kw_breathless         : 2,974 patients (29.7%)
  kw_no_symptoms        : 2,644 patients (26.4%)
  kw_wheezing           : 0 patients (0.0%)
  keyword_score        : mean = 0.85, max = 3


---
## 9. Merge Feature Groups <a id='9'></a>

All feature groups are merged on `id_paciente`. The analysis population is **restricted to patients with spirometry records**. These are the only patients with a valid gold-standard label.

All merges are anchored on `spiro_agg` (the labelled population) using a left join, so only patients with a valid spirometry label are retained.

In [35]:
# ── Start from labelled spirometry patients ────────────────────────────────────
df = spiro_agg[['id_paciente', 'copd_label']].copy()
print(f'Starting population: {len(df):,} patients (spirometry records only)')

# ── Merge Section A: original variables ───────────────────────────────────────
df = df.merge(patients_sel,  on='id_paciente', how='left')
df = df.merge(habits_sel,    on='id_paciente', how='left')
df = df.merge(labs_sel,      on='id_paciente', how='left')

# ── Merge Section B1: spirometry features ─────────────────────────────────────
df = df.merge(spiro_features, on='id_paciente', how='left')

# ── Merge Section B2: habits/labs engineered ──────────────────────────────────
df = df.merge(hl_features, on='id_paciente', how='left')

# ── Merge Section B3: ICD-10 flags ────────────────────────────────────────────
df = df.merge(icd_flags, on='id_paciente', how='left')

# ── Merge Section B4: NLP features ────────────────────────────────────────────
df = df.merge(nlp_features, on='id_paciente', how='left')

print(f'After merge         : {df.shape[0]:,} rows x {df.shape[1]:,} columns')
print(f'copd_label=1        : {df["copd_label"].sum():,}')
print(f'copd_label=0        : {(df["copd_label"]==0).sum():,}')

Starting population: 9,948 patients (spirometry records only)
After merge         : 9,948 rows x 39 columns
copd_label=1        : 7,600
copd_label=0        : 2,348


---
## 10. Missing Value Analysis <a id='10'></a>

Quantify missingness before imputation. Variables with more than 50% missing values are flagged. At that level, median imputation would replace the majority of observations with a synthetic value, potentially distorting the feature's signal. Such variables would be candidates for removal before training.

In [36]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'missing_n': missing, 'missing_pct': missing_pct})
missing_report = missing_report[missing_report['missing_n'] > 0].sort_values('missing_pct', ascending=False)

if len(missing_report) == 0:
    print('No missing values found.')
else:
    print(f'Variables with missing values ({len(missing_report)} of {df.shape[1]-1} features):')
    print(missing_report.to_string())

    # Threshold warning
    high_miss = missing_report[missing_report['missing_pct'] > 50]
    if len(high_miss) > 0:
        print(f'\nWARNING: {len(high_miss)} variable(s) with >50% missingness, consider dropping:')
        print(high_miss.to_string())

Variables with missing values (7 of 38 features):
                        missing_n  missing_pct
spiro_slope                  3375        33.93
vitamina_d_ng_ml             1019        10.24
colesterol_total_mg_dl       1018        10.23
pcr_mg_l                      995        10.00
ferritina_ng_ml               966         9.71
grupo_sanguineo               965         9.70
tsh_ui_ml                     925         9.30


---
## 11. Imputation and Encoding <a id='11'></a>

**Strategy:**
- **Numeric features:** Median imputation, robust to outliers and appropriate for skewed clinical distributions.
- **Categorical features:** 
  - `sexo`: Label encoded (H=0, M=1)
  - `actividad_fisica`: Ordinal encoded (`baja`=0, `media`=1, `alta`=2), matching values present in the raw data
- **Binary flags (ICD, NLP):** Already 0/1; missing values filled with 0 (no record = not diagnosed).

Note: Scaling (StandardScaler) is applied in the training script, not here, so the scaler can be fit on the training fold only. This prevents data leakage across CV folds.

In [37]:
df_proc = df.copy()

# ── Binary flags: missing -> 0 ────────────────────────────────────────────────
binary_cols = (
    ['fumador_actual', 'exfumador', 'ever_smoked', 'crp_elevated', 'vitamin_d_deficient'] +
    list(ICD_MAP.values()) +
    list(KEYWORD_PATTERNS.keys()) +
    ['keyword_score', 'n_unique_diag']
)
for col in binary_cols:
    if col in df_proc.columns:
        df_proc[col] = df_proc[col].fillna(0)

# ── Ordinal encoding: actividad_fisica ────────────────────────────────────────
# Values in data: 'baja', 'media', 'alta'
ACTIVITY_MAP = {'baja': 0, 'media': 1, 'alta': 2}
if 'actividad_fisica' in df_proc.columns:
    df_proc['actividad_fisica'] = (
        df_proc['actividad_fisica']
        .str.lower().str.strip()
        .map(ACTIVITY_MAP)
    )
    n_unmapped = df_proc['actividad_fisica'].isnull().sum()
    print(f'actividad_fisica encoded: {ACTIVITY_MAP}')
    print(f'  Unmapped (NaN) after encoding: {n_unmapped}')

# ── Label encoding: sexo ──────────────────────────────────────────────────────
if 'sexo' in df_proc.columns:
    sexo_vals = df_proc['sexo'].str.strip().str.upper().unique()
    print(f'\nsexo unique values: {sexo_vals}')
    df_proc['sexo'] = (df_proc['sexo'].str.strip().str.upper() == 'M').astype(float)
    print('  Encoded: H/Hombre=0, M/Mujer=1')

# ── Numeric: median imputation ────────────────────────────────────────────────
numeric_cols = df_proc.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['id_paciente', 'copd_label']]

imputation_log = {}
for col in numeric_cols:
    n_missing = df_proc[col].isnull().sum()
    if n_missing > 0:
        median_val = df_proc[col].median()
        df_proc[col] = df_proc[col].fillna(median_val)
        imputation_log[col] = {'n_imputed': n_missing, 'median': median_val}

if imputation_log:
    print(f'\nMedian imputation applied to {len(imputation_log)} numeric columns:')
    for col, info in imputation_log.items():
        print(f'  {col:<30}: {info["n_imputed"]:>4} values -> median = {info["median"]:.4f}')
else:
    print('\nNo numeric imputation needed.')

# ── One-hot encoding: grupo_sanguineo (sensitivity analysis) ──────────────────
# Reference category: O (most common in Spain ~44%) -- dropped to avoid multicollinearity
if 'grupo_sanguineo' in df_proc.columns:
    bg_vals = df_proc['grupo_sanguineo'].str.strip().str.upper().unique()
    print(f'\ngrupo_sanguineo unique values: {bg_vals}')
    df_proc['grupo_sanguineo'] = df_proc['grupo_sanguineo'].str.strip().str.upper()
    bg_dummies = pd.get_dummies(df_proc['grupo_sanguineo'], prefix='bg', drop_first=False)
    # Keep A, B, AB; drop O as reference
    for col in ['bg_A', 'bg_B', 'bg_AB']:
        if col in bg_dummies.columns:
            df_proc[col] = bg_dummies[col].astype(int)
        else:
            df_proc[col] = 0  # category not present in data
    df_proc.drop(columns=['grupo_sanguineo'], inplace=True)
    print('  One-hot encoded: bg_A, bg_B, bg_AB (reference = O)')
    print(f'  bg_A : {df_proc["bg_A"].sum():,}  bg_B : {df_proc["bg_B"].sum():,}  bg_AB: {df_proc["bg_AB"].sum():,}')

# ── Verify no remaining missing values ────────────────────────────────────────
remaining_missing = df_proc.drop(columns=['id_paciente']).isnull().sum().sum()
print(f'\nRemaining missing values after imputation: {remaining_missing}')

actividad_fisica encoded: {'baja': 0, 'media': 1, 'alta': 2}
  Unmapped (NaN) after encoding: 0

sexo unique values: <ArrowStringArray>
['F', 'M']
Length: 2, dtype: str
  Encoded: H/Hombre=0, M/Mujer=1

Median imputation applied to 6 numeric columns:
  pcr_mg_l                      :  995 values -> median = 2.5600
  vitamina_d_ng_ml              : 1019 values -> median = 24.1000
  colesterol_total_mg_dl        : 1018 values -> median = 188.6000
  ferritina_ng_ml               :  966 values -> median = 119.0000
  tsh_ui_ml                     :  925 values -> median = 2.0900
  spiro_slope                   : 3375 values -> median = 0.0031

grupo_sanguineo unique values: <ArrowStringArray>
['AB', 'B', 'A', 'O', nan]
Length: 5, dtype: str
  One-hot encoded: bg_A, bg_B, bg_AB (reference = O)
  bg_A : 2,323  bg_B : 2,269  bg_AB: 2,197

Remaining missing values after imputation: 0


---
## 12. Final Feature Matrix Validation <a id='12'></a>

Final checks before saving:
1. Confirm no excluded variables (leakage sources, bias proxies) have slipped into the feature set
2. Confirm `copd_label` is correctly distributed
3. Print the complete feature list with data types and a per-feature missingness check

In [38]:
# ── Define final feature columns (exclude ID and label) ──────────────────────
EXCLUDED_VARS = [
    'spiro_min_ratio',      # label leakage
    'spiro_first_ratio',    # absorbed into slope
    'spiro_first_date',     # absorbed into years_span
    'spiro_last_date',      # absorbed into years_span
    'epoc_diagnostico',     # label leakage (administrative)
    'has_copd_diag',        # label leakage (ICD J44)
    'nivel_socioeconomico', # bias risk
    'zona_residencia',      # bias risk
    # grupo_sanguineo is not listed here -- it was one-hot encoded into bg_A/bg_B/bg_AB above
    # bg_* columns are included in features.csv; excluded from FEATURES_MAIN in train_model_v2.py
    'id_paciente',          # identifier
    'copd_label',           # this is the label, not a feature
]

FEATURE_COLS = [c for c in df_proc.columns if c not in EXCLUDED_VARS]

# ── Leakage check: confirm none of the excluded vars slipped through ──────────
leakage_check = [v for v in EXCLUDED_VARS if v in FEATURE_COLS]
if leakage_check:
    print(f'ERROR: excluded variables found in feature set: {leakage_check}')
else:
    print('Leakage check PASSED: no excluded variables in feature set.')

print(f'\nFinal feature matrix: {len(df_proc):,} rows x {len(FEATURE_COLS)} features')
print(f'Label distribution  : {df_proc["copd_label"].value_counts().to_dict()}')
print(f'\nFeature list ({len(FEATURE_COLS)} features):')
for i, col in enumerate(FEATURE_COLS, 1):
    dtype = df_proc[col].dtype
    n_missing = df_proc[col].isnull().sum()
    missing_str = f'  *** {n_missing} MISSING ***' if n_missing > 0 else ''
    print(f'  {i:>2}. {col:<35} [{dtype}]{missing_str}')

Leakage check PASSED: no excluded variables in feature set.

Final feature matrix: 9,948 rows x 39 features
Label distribution  : {1: 7600, 0: 2348}

Feature list (39 features):
   1. edad                                [int64]
   2. sexo                                [float64]
   3. imc                                 [float64]
   4. fumador_actual                      [int64]
   5. exfumador                           [int64]
   6. paquetes_ano                        [float64]
   7. actividad_fisica                    [int64]
   8. pcr_mg_l                            [float64]
   9. vitamina_d_ng_ml                    [float64]
  10. colesterol_total_mg_dl              [float64]
  11. ferritina_ng_ml                     [float64]
  12. tsh_ui_ml                           [float64]
  13. spiro_latest_ratio                  [float64]
  14. spiro_slope                         [float64]
  15. spiro_n_tests                       [int64]
  16. spiro_years_span                    [float64]


---
## 13. Save Outputs <a id='13'></a>

Two files saved to `data/processed/`:
- **`features.csv`**: feature matrix, one row per patient, `id_paciente` included as index anchor
- **`labels.csv`**: label vector with `id_paciente` for alignment verification in the training script

In [39]:
# ── Features ──────────────────────────────────────────────────────────────────
features_out = df_proc[['id_paciente'] + FEATURE_COLS].copy()
features_path = PROC / 'features.csv'
features_out.to_csv(features_path, index=False)

# ── Labels ────────────────────────────────────────────────────────────────────
labels_out = df_proc[['id_paciente', 'copd_label']].copy()
labels_path = PROC / 'labels.csv'
labels_out.to_csv(labels_path, index=False)

print('Saved:')
print(f'  features.csv : {features_out.shape[0]:,} rows x {features_out.shape[1]:,} cols  ->  {features_path}')
print(f'  labels.csv   : {labels_out.shape[0]:,} rows x {labels_out.shape[1]:,} cols   ->  {labels_path}')

# ── Alignment check ───────────────────────────────────────────────────────────
assert (features_out['id_paciente'].values == labels_out['id_paciente'].values).all(), \
    'ERROR: id_paciente order mismatch between features and labels!'
print('\nAlignment check PASSED: features and labels are in the same row order.')

# ── Summary ───────────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('PREPROCESSING COMPLETE')
print('='*60)
print(f'Training population : {len(features_out):,} patients (spirometry-confirmed)')
print(f'Features            : {len(FEATURE_COLS)}')
print(f'COPD positive       : {labels_out["copd_label"].sum():,} ({labels_out["copd_label"].mean()*100:.1f}%)')
print(f'COPD negative       : {(labels_out["copd_label"]==0).sum():,} ({(1-labels_out["copd_label"].mean())*100:.1f}%)')
print(f'\nNext step: train_model_v2.py')

Saved:
  features.csv : 9,948 rows x 40 cols  ->  c:\Users\jorge\OneDrive - IE University\Documentos\MBDS2025\Biopharma Specialization\Project\Dev\data\processed\features.csv
  labels.csv   : 9,948 rows x 2 cols   ->  c:\Users\jorge\OneDrive - IE University\Documentos\MBDS2025\Biopharma Specialization\Project\Dev\data\processed\labels.csv

Alignment check PASSED: features and labels are in the same row order.

PREPROCESSING COMPLETE
Training population : 9,948 patients (spirometry-confirmed)
Features            : 39
COPD positive       : 7,600 (76.4%)
COPD negative       : 2,348 (23.6%)

Next step: train_model_v2.py
